# Sahayak Triage — Phase 1: Data Extraction and Cleaning

This notebook loads the raw Yale ED Triage and Admission dataset (`5v_cleandf.rdata`), filters it for patients presenting with fever and infection chief complaints, handles missing values, and saves the cleaned dataset for model training.

## 1. Import Packages

In [1]:
import pyreadr
import pandas as pd
import os
import gc

## 2. Load Raw RData File

In [2]:
raw_path = "../data/raw/5v_cleandf_subset.csv"
print("Loading subsetted raw CSV file...")
df = pd.read_csv(raw_path)
print(f"Raw data loaded. Shape: {df.shape}")

Loading subsetted raw CSV file...
Raw data loaded. Shape: (560486, 25)


## 3. Whitelist Columns (Prevent Target Leakage)

To prevent target leakage, we only keep the clinical indicators (vitals, chief complaints, demographics) available at intake. We drop all outcome variables such as department disposition and admission status.

In [3]:
target_col = 'esi'

demographic_cols = ['age', 'gender']

vital_cols = [
    'triage_vital_hr', 
    'triage_vital_sbp', 
    'triage_vital_dbp', 
    'triage_vital_rr', 
    'triage_vital_o2', 
    'triage_vital_temp'
]

fever_infection_cc = [
    'cc_breathingdifficulty', 
    'cc_breathingproblem', 
    'cc_chills', 
    'cc_coldlikesymptoms',
    'cc_cough', 
    'cc_fever', 
    'cc_fever-75yearsorolder', 
    'cc_fever-9weeksto74years',
    'cc_feverimmunocompromised', 
    'cc_nasalcongestion', 
    'cc_respiratorydistress',
    'cc_shortnessofbreath', 
    'cc_sorethroat', 
    'cc_unresponsive', 
    'cc_urinarytractinfection', 
    'cc_woundinfection'
]

all_selected_cols = [target_col] + demographic_cols + vital_cols + fever_infection_cc

df = df[all_selected_cols]
gc.collect()
print(f"Subsetted columns. Shape: {df.shape}")

Subsetted columns. Shape: (560486, 25)


## 4. Row Filtering (Filter for Fever & Infection presentations)

In [4]:
print("Filtering rows for fever/infection chief complaints...")
filter_condition = pd.Series(False, index=df.index)
for col in fever_infection_cc:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    filter_condition = filter_condition | (df[col] == 1)
    
df_filtered = df[filter_condition].copy()
print(f"Filtered to fever/infection cases. Shape: {df_filtered.shape}")

Filtering rows for fever/infection chief complaints...
Filtered to fever/infection cases. Shape: (64308, 25)


## 5. Cleaning Target Variable and Features

In [5]:
print("Cleaning target variable (ESI)...")
df_filtered = df_filtered.dropna(subset=[target_col])
df_filtered[target_col] = df_filtered[target_col].astype(int)
print(f"Dropped missing target rows. Shape: {df_filtered.shape}")

print("Cleaning demographics...")
# Mapped MALE -> 1, FEMALE -> 0, others -> -1
df_filtered['gender'] = df_filtered['gender'].astype(str).str.upper().map({'MALE': 1, 'FEMALE': 0}).fillna(-1).astype(int)

# Age: numeric, fill missing with median
df_filtered['age'] = pd.to_numeric(df_filtered['age'], errors='coerce')
median_age = df_filtered['age'].median()
df_filtered['age'] = df_filtered['age'].fillna(median_age).astype(float)

print("Cleaning triage vitals...")
for col in vital_cols:
    df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce')
    
print("Engineering clinical features...")
# Missingness indicators for vitals
for col in vital_cols:
    df_filtered[f'{col}_is_missing'] = df_filtered[col].isnull().astype(int)
    
# Clinical scores (use clinical defaults to compute scores without modifying the raw features)
hr = df_filtered['triage_vital_hr'].fillna(70)
sbp = df_filtered['triage_vital_sbp'].fillna(120)
dbp = df_filtered['triage_vital_dbp'].fillna(80)
rr = df_filtered['triage_vital_rr'].fillna(15)
o2 = df_filtered['triage_vital_o2'].fillna(98)
temp = df_filtered['triage_vital_temp'].fillna(98.6)

# 1. qSOFA
df_filtered['qsofa_rr_high'] = (rr >= 22).astype(int)
df_filtered['qsofa_sbp_low'] = (sbp <= 100).astype(int)
df_filtered['qsofa_altered_mental'] = df_filtered['cc_unresponsive']
df_filtered['qsofa_score'] = df_filtered['qsofa_rr_high'] + df_filtered['qsofa_sbp_low'] + df_filtered['qsofa_altered_mental']

# 2. SIRS
df_filtered['sirs_temp_abnormal'] = ((temp > 100.4) | (temp < 96.8)).astype(int)
df_filtered['sirs_hr_high'] = (hr > 90).astype(int)
df_filtered['sirs_rr_high'] = (rr > 20).astype(int)
df_filtered['sirs_score'] = df_filtered['sirs_temp_abnormal'] + df_filtered['sirs_hr_high'] + df_filtered['sirs_rr_high']

# 3. Shock Index and Pulse Pressure
df_filtered['shock_index'] = hr / sbp
df_filtered['pulse_pressure'] = sbp - dbp

# 4. Hypoxia indicators
df_filtered['hypoxia_severe'] = (o2 < 90).astype(int)
df_filtered['hypoxia_moderate'] = (o2 < 95).astype(int)

# 5. Age elderly
df_filtered['age_elderly'] = (df_filtered['age'] >= 65).astype(int)

print("Cleaning and feature engineering completed.")

Cleaning target variable (ESI)...
Dropped missing target rows. Shape: (64132, 25)
Cleaning demographics...
Cleaning triage vitals...
Engineering clinical features...
Cleaning and feature engineering completed.


## 6. Save Cleaned Dataset

In [6]:
processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)
output_path = os.path.join(processed_dir, "cleaned_triage_data.csv")
print(f"Saving cleaned data to {output_path}...")
df_filtered.to_csv(output_path, index=False)
print("Data cleaning completed successfully!")

Saving cleaned data to ../data/processed\cleaned_triage_data.csv...
Data cleaning completed successfully!


## 7. Basic Data Exploration & Target Distribution

In [7]:
print("Cleaned Data Info:")
print(df_filtered.info())

print("\nTarget Variable (ESI) Urgency Distribution:")
print(df_filtered[target_col].value_counts().sort_index())

Cleaned Data Info:
<class 'pandas.DataFrame'>
Index: 64132 entries, 14 to 560480
Data columns (total 44 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   esi                           64132 non-null  int64  
 1   age                           64132 non-null  float64
 2   gender                        64132 non-null  int64  
 3   triage_vital_hr               42158 non-null  float64
 4   triage_vital_sbp              41825 non-null  float64
 5   triage_vital_dbp              41806 non-null  float64
 6   triage_vital_rr               41559 non-null  float64
 7   triage_vital_o2               32855 non-null  float64
 8   triage_vital_temp             40410 non-null  float64
 9   cc_breathingdifficulty        64132 non-null  int64  
 10  cc_breathingproblem           64132 non-null  int64  
 11  cc_chills                     64132 non-null  int64  
 12  cc_coldlikesymptoms           64132 non-null  int64  
 